# Can you tell who is about to get diabetes by looking at their gut bacteria?

Forty trillion bacteria live in your large intestine. They ferment the fibre you
cannot digest into short-chain fatty acids, and one of those - **butyrate** - fuels
your colon cells, strengthens the gut barrier, and has been linked to how sensitively
your tissues respond to insulin.

So: if a gut community shifts away from the bacteria that make butyrate, does the
person drift toward type 2 diabetes? And could a stool sample say so **before** the
blood test does?

Blood glucose tells you where somebody is now. It is much worse at telling you
where somebody is heading.

## The study

Five centres enrolled **700 adults with prediabetes** - blood sugar above normal,
below the line where anyone would call it diabetes. Each gave one fasting blood
sample and one stool sample at enrolment. All were followed for twelve months to
see who crossed the diagnostic line.

Two questions:

1. Given what we knew on the day someone enrolled, can we say whether they will be
   diagnosed within the year?
2. **Does the stool sample add anything the blood test had not already said?**

## How to work

Run the cells in order. Where you see **Task**, the cell below it is yours to
write - there is a hint from last week's notebook in the comment. Where a cell is
marked *supplied*, just run it.

If you get stuck, skip the task and keep going. The notebook still runs.

**In Google Colab:** use a Python CPU runtime. The setup cell installs the required packages, and the data load directly from GitHub. Use **File → Save a copy in Drive** to keep your work, then download the completed `.ipynb` for submission.

[Session 3 handout](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-03-t2d-microbiome/notebooks/session_03/week03_microbiome_handout.md).


## 1 · Setup  *(supplied)*

In [ ]:
%pip install -q numpy pandas scikit-learn


In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_PATH = "https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-03-t2d-microbiome/notebooks/session_03/data/t2d_microbiome_week03.csv"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("Ready.")

## 2 · Meet the table

### Task 1
Load the file into `mb`. Print its shape, the first rows, the number of distinct
`participant_id` values, and the share of each value in `progressed_12m`.

In [ ]:
# TASK 1
# Hint from last week:
#   ed = pd.read_csv(...)
#   print(ed.shape);  print(ed.head())
#   print(ed["patient_id"].nunique())
#   print(ed["deterioration_24h"].value_counts(normalize=True).round(3))

mb = pd.read_csv(DATA_PATH)


## 3 · Look properly

Six checks. Each is one or two lines. Print, do not interpret yet.

### Task 2 - types
Print the dtype of every column. Then print the first few values of the column
that looks wrong.

In [ ]:
# TASK 2
# Hint: last week  print(ed.dtypes)



### Task 3 - missing values
Print the percentage missing for every column that has any, sorted largest first.
Two columns are missing on exactly the same rows - why?

In [ ]:
# TASK 3
# Hint: last week
#   missing = (ed.isna().mean() * 100).round(1)
#   print(missing[missing > 0].sort_values(ascending=False))



### Task 4 - distinct values
For every text column, print how many distinct values it takes. Two columns have
more values than they have real categories.

In [ ]:
# TASK 4
# Hint: last week
#   for col in ed.select_dtypes(include=["object", "string"]).columns:
#       print(col, ed[col].nunique())



### Task 5 - by site
Print the median of `hba1c`, `waist_cm` and `fasting_glucose_mmol_l` for each
site. Then cross-tabulate `site` against `extraction_kit`.

In [ ]:
# TASK 5
# Hint: last week  ed.groupby("site")[["creatinine", ...]].median()
#       and        pd.crosstab(a, b)



### Task 6 - each column on its own
Score every numeric column as if it were the whole model: fill its gaps with the
median, compute the ROC-AUC against `progressed_12m`, and keep the larger of
`auc` and `1 - auc`. Sort and print the top ten.

The top column beats every biological measurement in the table. By how much - and
does that make you pleased, or suspicious?

In [ ]:
# TASK 6
# Hint: last week's exercise 4
#   scores = {}
#   for col in <numeric columns except the target>:
#       values = ed[col].fillna(ed[col].median())
#       auc = roc_auc_score(ed["deterioration_24h"], values)
#       scores[col] = max(auc, 1 - auc)
#   print(pd.Series(scores).sort_values(ascending=False).round(3))



**Checkpoint 1** - hands up when you have three findings and one column you will not use.

## 4 · The data dictionary

Your instructor has a printed copy. You can also [open the data dictionary on GitHub](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-03-t2d-microbiome/notebooks/session_03/data/t2d_microbiome_DICTIONARY.md). Read it against the column list.

### Task 7
The dictionary says nobody taking metformin was eligible. Count the rows with
`metformin_initiated == 1`, and cross-tabulate it against the outcome. What does
the contradiction tell you about when that column was filled in?

### Task 8
`antibiotics_last_3m` was known at enrolment, so it is a different kind of
problem. But antibiotics disturb the gut for reasons unrelated to diabetes. Count
how many participants had a course. What would you do about them?

In [ ]:
# TASKS 7 and 8
# Hint:  (mb["col"] == 1).sum()      pd.crosstab(mb["a"], mb["b"])



## 5 · Clean what is broken

### Task 9 - a number stored as text
Convert `bmi` to numeric. First try `pd.to_numeric(..., errors="coerce")` directly
and count how many values it loses. Then fix the real problem and convert again.
**Always print how many values a conversion lost.**

In [ ]:
# TASK 9
# Hint: last week, for temp_c:
#   naive = pd.to_numeric(ed["temp_c"], errors="coerce");  print(naive.isna().sum())
#   ed["temp_c"] = pd.to_numeric(ed["temp_c"].astype(str).str.replace(",", ".", regex=False),
#                                errors="coerce")



### Task 10 - one measurement, two units
Make a new column `hba1c_pct` that is `hba1c` converted to percent at the site
that reports mmol/mol, and unchanged elsewhere. IFCC to DCCT is
`pct = mmol / 10.929 + 2.15`. Print the median by site afterwards.

In [ ]:
# TASK 10
# Hint: last week, for creatinine:
#   ed["creatinine"] = ed["creatinine"].where(ed["site"] != "C", ed["creatinine"] / 88.4)
#   or with np.where(condition, value_if_true, value_if_false)



### Task 11 - spellings
Make `sex_c` with two values and `smoking_c` with three. Print the value counts
of each to confirm.

In [ ]:
# TASK 11
# Hint: last week  ed["sex"] = ed["sex"].str.upper().str[0]
#       and        .str.lower().str.replace("old", "new", regex=False)



## 6 · Your column decisions

### Task 12
List every column you will **not** use as a feature. One per line, with a reason
in the comment. Think about: identifiers, constants, columns you replaced, columns
computable from others, and anything that could not have been known at enrolment.

In [ ]:
# TASK 12
DROP_COLS = [
    "participant_id",     # unique per row
    # add yours below
]


*Supplied.* Splits the remaining columns by type.

In [ ]:
TARGET = "progressed_12m"
features = [c for c in mb.columns if c not in DROP_COLS + [TARGET]]
NUMERIC = [c for c in features if pd.api.types.is_numeric_dtype(mb[c])]
ORDERED = [c for c in features if c == "physical_activity"]
NOMINAL = [c for c in features if c not in NUMERIC + ORDERED]
print(f"{len(NUMERIC)} numeric, {len(NOMINAL)} nominal, {len(ORDERED)} ordered")
print("nominal:", NOMINAL)

## 7 · The pipeline  *(supplied)*

This is the object you were assembling by hand last week. Every step inside it is
fitted on the training rows only, every fold, without anyone remembering to.

Look at the ranges printed below. The `clr_` columns arrive already transformed
and share a scale. The clinical columns do not.

In [ ]:
numeric_steps = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                          ("scale",  StandardScaler())])
nominal_steps = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))])
ordered_steps = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OrdinalEncoder(categories=[["low","moderate","high"]],
                                     handle_unknown="use_encoded_value", unknown_value=-1))])

preprocessor = ColumnTransformer([("num", numeric_steps, NUMERIC),
                                  ("nom", nominal_steps, NOMINAL),
                                  ("ord", ordered_steps, ORDERED)])

def build(model):
    return Pipeline([("prep", preprocessor), ("model", model)])

X, y = mb[features], mb[TARGET]
print(mb[["waist_cm", "fasting_insulin_mu_l", "clr_Roseburia"]]
        .describe().loc[["min", "max"]].round(2))

## 8 · Fit it

### Task 13
Write down the AUC you expect. Then cross-validate `build(model)` with stratified
5-fold for four models - logistic regression, a random forest, k-NN, and a small
neural network (`MLPClassifier(hidden_layer_sizes=(32,), max_iter=1500)`) - and
print each mean ROC-AUC.

In [ ]:
# TASK 13
# Hint: last week
#   cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
#   cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc").mean()



**Checkpoint 2** - hands up when you have a number you would defend.

## 9 · Is it real?

### Task 14 - how much does the score move on its own?
Fit the *same* logistic pipeline on 15 different 75/25 stratified splits and
collect the test AUC each time. Print the minimum, the maximum and the range.

In [ ]:
# TASK 14
# Hint: last week
#   for s in range(15):
#       tr, te = next(StratifiedShuffleSplit(1, test_size=.25, random_state=s).split(X, y))
#       fit = build(model).fit(X.iloc[tr], y.iloc[tr])
#       auc = roc_auc_score(y.iloc[te], fit.predict_proba(X.iloc[te])[:, 1])



### Task 15 - what did the gut add?
Define `TAXA` as the features starting with `clr_` and `CLIN` as the other numeric
features. Cross-validate a logistic pipeline on clinical only, taxa only, and both.
Compare the gain against the range from Task 14.

In [ ]:
# TASK 15
# Hint: build a small pipeline on a subset of columns
#   pre = ColumnTransformer([("num", numeric_steps, cols)])
#   pipe = Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=4000))])
#   cross_val_score(pipe, mb[cols], y, cv=cv, scoring="roc_auc").mean()

TAXA = [c for c in features if c.startswith("clr_")]
CLIN = [c for c in NUMERIC if not c.startswith("clr_")]


## 10 · Optional - would it work somewhere else?

Only if you have finished everything above.

### Task 16
Hold each site out in turn. Train a taxa-only pipeline and a clinical-only
pipeline on the other four sites, score both on the held-out one, and print the
two AUCs beside the site's `extraction_kit`. Two sites do clearly worse on taxa.
What do they have in common?

In [ ]:
# TASK 16
# Hint:  for s in sorted(mb["site"].unique()):
#            tr, te = mb["site"] != s, mb["site"] == s
#            ... fit on mb.loc[tr, cols], score on mb.loc[te, cols]



## 11 · Your report

Fill in the table below, then **upload this notebook to Moodle**. It is the
deliverable - not a separate document.

| | |
|---|---|
| What was wrong with the table | |
| What I dropped, and why | |
| My score | |
| Is it bigger than the spread? | |
| What the microbiome added | |
| One thing I could not answer from the file | |

**Checkpoint 3** - hands up when every row has something in it, then upload.